In [6]:
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path

In [18]:
def load_csv(filename):
    candidates = [Path(filename), Path.cwd() / filename]
    print(candidates)
    candidates.extend(Path.cwd().rglob(filename))

    seen = set()
    for path in candidates:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if path.exists():
            return pd.read_csv(path)

    raise FileNotFoundError(f"{filename} 파일을 찾지 못했어요.")
    

In [8]:
def load_tips():
    try:
        return sns.load_dataset("tips")
    except Exception:
        return load_csv("tips.csv")

In [4]:
def load_telco():
    filenames = [
        "WA_Fn-UseC_-Telco-Customer-Churn.csv",
        "telco_churn.csv",
        "Telco-Customer-Churn.csv",
    ]

    for filename in filenames:
        try:
            return load_csv(filename)
        except FileNotFoundError:
            continue

    raise FileNotFoundError("Telco Churn CSV 파일을 찾지 못했어요.")

## 문제 1. [쉬움] Tips 데이터에서 DataFrame 구조 파악하기
### 문제 설명

Tips도 결국 하나의 DataFrame이에요.
`shape`, `columns`, `dtypes`, `index`를 직접 확인하면서 표 구조를 읽어볼게요.
필요하면 특정 열 하나를 꺼내 Series인지 같이 확인해봐도 좋아요.

In [22]:
# TODO: tips 데이터를 불러와요.
tips = sns.load_dataset('tips')
# TODO: tips의 shape를 출력해요.
print('###tips Shape :', tips.shape)

# TODO: tips의 columns를 리스트로 출력해요.
print('###tips column :',tips.columns.to_list() )

# TODO: tips의 dtypes를 출력해요.
print('###tips dtypes : ', tips.dtypes)

# TODO: tips의 index를 출력해요.
print('###tips index', tips.index)

# TODO: tips의 앞 3행을 출력해서 표 구조를 눈으로 확인해요.
print('tips head', tips.head(3))

# TODO: 선택 사항이에요. tips["total_bill"]을 꺼내서 type()으로 확인해요.
print('tips["total_bill"].type()', type(tips["total_bill"])) 

###tips Shape : (244, 7)
###tips column : ['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']
###tips dtypes :  total_bill     float64
tip            float64
sex           category
smoker        category
day           category
time          category
size             int64
dtype: object
###tips index RangeIndex(start=0, stop=244, step=1)
tips head    total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
2       21.01  3.50    Male     No  Sun  Dinner     3
tips["total_bill"].type() <class 'pandas.Series'>


## 문제 2. [쉬움] Tips 데이터에서 loc/iloc 선택과 조건 필터링 해보기

### 문제 설명

이번에는 Tips에서 원하는 행과 열을 직접 골라볼게요.
`loc`, `iloc`, 조건 필터링, `str.contains()`, `drop()`, `rename()`까지 한 번에 연결해보면 Day 1 흐름이 잘 정리돼요.

In [42]:
# TODO: tips 데이터를 다시 불러와요.
tips = sns.load_dataset('tips')

# TODO: loc으로 0~4번 행의 total_bill, tip, time 열만 선택해요.
#loc[행, 열] : 라벨 기준(끝 인덱스 포함)
tips.loc[0:4, ['total_bill', 'tip', 'time']]

# TODO: iloc으로 앞 5행, 앞 3열을 선택해요.
#iloc : 인덱스 기준(끝 인덱스 제와 0:5 -> 0~4)
tips.iloc[0:5,0:3]

# TODO: total_bill이 20 이상인 행만 필터링해요.
tips[tips['total_bill'] >= 20]


# TODO: time 열에 "Din"이 들어가는 행만 str.contains()로 필터링해요.
tips[tips['time'].str.contains('Din', na=False)]


# TODO: smoker가 "Yes"이면서 day가 "Sun"인 행만 필터링해요.
new_tips = tips[(tips["smoker"] == "Yes") & (tips["day"] == "Sun")]    


# TODO: 위 결과에서 size 열을 drop()으로 제거해요.
new_tips = new_tips.drop(columns=['size'])

# TODO: total_bill -> total_bill_usd, tip -> tip_usd로 rename() 해요.
result = new_tips
result = result.rename(columns={'total_bill': 'total_bill_usd', 'tip':'tip_u'})

# TODO: 최종 결과의 앞 5행을 출력해요.
result.head(5)


,total_bill_usd,tip_u,sex,smoker,day,time
164,17.51,3.00,Female,Yes,Sun,Dinner
172,7.25,5.15,Male,Yes,Sun,Dinner
173,31.85,3.18,Male,Yes,Sun,Dinner
174,16.82,4.00,Male,Yes,Sun,Dinner
175,32.90,3.11,Male,Yes,Sun,Dinner


## 문제 3. [보통] Tips 데이터 결측치 진단하고 dropna/fillna 처리하기

### 문제 설명

Tips 원본에는 결측치가 거의 없을 수 있어서, 연습용 복사본에 일부 결측을 만들어서 처리해볼게요.
`isnull()`, `dropna()`, `fillna()` 흐름을 손으로 직접 해보는 게 핵심이에요.

In [15]:
# TODO: tips 데이터를 불러온 뒤 practice_tips라는 복사본을 만들어요.
tips = load_tips()
practice_tips = tips.copy()


# TODO: practice_tips의 일부 행에 결측치를 넣어요.
# TODO: 예시로 total_bill 1개, tip 1개, day 1개 정도를 loc으로 비워 보세요.
practice_tips.loc[0,'total_bill'] = np.nan
practice_tips.loc[1,'tip']  = np.nan
practice_tips.loc[2,'day'] = np.nan
practice_tips.head()


# TODO: 열별 결측 개수를 isnull().sum()으로 출력해요.
practice_tips.isnull().sum()

# TODO: 결측치가 있는 행만 따로 확인해요.
# 필터와 집계는 다르다. 필터는 대괄호에 안 넣어도 되지만, 이 문제는 필터링이므로 df[조건]형태로 코드를짜야한다.
practice_tips[practice_tips.isnull().any(axis=1)]

# TODO: total_bill 또는 tip이 비어 있는 행은 dropna(subset=...)로 제거해요.
#subset에 열 이름을 지정하면 그 열들 중 하나라도 결측이면 행을 제거한다.
practice_tips = practice_tips.dropna(subset=['total_bill', 'tip'])

# TODO: day 열 결측치는 fillna()로 "Unknown"으로 채워요.
#fillna()는 새로운 Series를 반환하기 때문에 왼쪽에 다시 대입을 해야 실제로 반영이 된다.
# practice_tips["day"] = practice_tips['day'].fillna('Unknown')
#TypeError: Cannot setitem on a Categorical with a new category (Unknown), set the categories first
# day열이 Categorical type이라 기존 카테고리에 없는 값은 바로 넣을 수 없어서, 문자열로 변환한 뒤 채우면 된다.
practice_tips['day'] = practice_tips['day'].astype(str).replace('nan', 'Unknown')

# TODO: 처리 후 결측 개수를 다시 출력해요.
practice_tips.isnull().sum()

# TODO: 처리 전 행 수와 처리 후 행 수를 비교해요.
print(f'처리 전 수 : {len(tips)} 처리 후 수 : {len(practice_tips)}')

처리 전 수 : 244 처리 후 수 : 242



## 문제 4. [보통] Telco CSV를 읽고 구조 파악 후 TotalCharges 타입 변환하기

### 문제 설명

이제 Day 1 관통예제였던 Telco 데이터로 넘어가 볼게요.
CSV를 읽고 구조를 파악한 뒤, `TotalCharges` 공백 문자열을 수치형으로 바꾸고 중복 행도 정리해볼게요.

In [38]:
# TODO: Telco CSV를 불러오고 telco라는 변수에 저장해요.
telco = pd.read_csv("../../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")   


# TODO: telco의 shape, columns 개수, dtypes 일부를 출력해요.
print(telco.shape, telco.columns, telco.dtypes.head(10))

# TODO: info()를 실행해서 TotalCharges dtype을 확인해요.
telco.info() # str


# TODO: TotalCharges 열에서 공백 문자열 개수를 세어 봐요.
빈문자열 = (telco['TotalCharges'].astype(str).str.strip() =="").sum()
print('빈문자열 : ', 빈문자열)


# TODO: 원본 보호를 위해 telco_clean = telco.copy()를 만들어요.
telco_clean = telco.copy()


# TODO: TotalCharges의 공백 문자열을 np.nan으로 바꿔요.
telco_clean['TotalCharges'] = telco_clean['TotalCharges'].replace(' ', np.nan)
print('isnull의 개수', telco_clean['TotalCharges'].isnull().sum())
# TODO: TotalCharges를 pd.to_numeric(errors="coerce")로 수치형으로 변환해요.
# 수치형으로 바꾸고 errors='coerce'는 변환 실패한 값을 NaN으로 바꾼다는 뜻이다.
# errors options -> raise => 오류 발생시킴, ignore -> 원래 값 그대로 유지
telco_clean['TotalCharges'] = pd.to_numeric(telco_clean['TotalCharges'], errors='coerce')


# TODO: 변환 후 TotalCharges dtype과 결측 개수를 출력해요.
telco_clean['TotalCharges'].dtype
print(telco_clean['TotalCharges'].isnull().sum())


# TODO: 전체 중복 행 수를 확인해요.
print('duplicated ')s
print(telco_clean.duplicated().sum())
# TODO: drop_duplicates()로 중복을 제거한 뒤 행 수 변화를 확인해요.
telco_clean.drop_duplicates()
before = len(telco_clean)                                                                                                                                                             
telco_clean = telco_clean.drop_duplicates()                                                                                                                                         
after = len(telco_clean)                                                                                                                                                              
print("제거 전:", before, "/ 제거 후:", after, "/ 제거된 행 수:", before - after)                                                                                                     
                                                                                         

(7043, 21) Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str') customerID           str
gender               str
SeniorCitizen      int64
Partner              str
Dependents           str
tenure             int64
PhoneService         str
MultipleLines        str
InternetService      str
OnlineSecurity       str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner    

## 문제 5. [도전] Telco에서 .copy() 패턴으로 파생 열 만들고 SettingWithCopyWarning 피하기

### 문제 설명

마지막 문제는 Day 1 핵심인 `.copy()` 패턴을 적용해보는 문제예요.
필터링한 부분집합에 바로 값을 넣지 말고, `.copy()`로 안전한 작업용 DataFrame을 만든 뒤 파생 열을 추가해볼게요.


In [ ]:
# TODO: Telco 데이터를 다시 불러오고 필요한 기본 정제를 먼저 해요.
# TODO: TotalCharges 공백 처리와 수치형 변환까지 끝낸 상태를 만들어요.
telco = pd.read_csv("../../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")   

In [40]:
                                                                                                               
telco_clean = telco.copy()                                                                                                                                                          
telco_clean["TotalCharges"] = telco_clean["TotalCharges"].replace(" ", np.nan)                                                                                                        
telco_clean["TotalCharges"] = pd.to_numeric(telco_clean["TotalCharges"], errors="coerce")  

In [41]:
telco_clean.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [43]:
# TODO: Contract가 "Month-to-month"인 고객만 골라서 month_df를 만들어요.
# TODO: month_df를 만들 때 .copy()를 붙여서 독립된 DataFrame으로 만들어요.
month_df = telco_clean[telco_clean['Contract'] == "Month-to-month"].copy()


In [50]:
#TODO: tenure가 0인 경우를 먼저 확인해요.
print((month_df['tenure'] == 0).sum())
month_df.head()

0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,No


In [52]:
month_df.loc[:,'charge_per_month'] = month_df["TotalCharges"] / month_df['tenure'].replace(0,np.nan)
# tenure가 0 인 경우 먼저 np.nan으로 바꿔서 0으로 나누는걸 방지

In [53]:
month_df.loc[:,'senior_partner'] =( month_df['SeniorCitizen'] == 1 )& (month_df['Partner'] == 'Yes')

In [54]:
month_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,charge_per_month,senior_partner
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,29.850000,False
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,54.075000,False
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,75.825000,False
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,102.562500,False
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,No,88.609091,False


In [56]:
                                                                                                                                                                                        
month_df[["customerID", "Contract", "tenure", "TotalCharges", "charge_per_month", "senior_partner"]].head()                                                                         
                                                                                                                 

,customerID,Contract,tenure,TotalCharges,charge_per_month,senior_partner
0,7590-VHVEG,Month-to-month,1,29.85,29.850000,False
2,3668-QPYBK,Month-to-month,2,108.15,54.075000,False
4,9237-HQITU,Month-to-month,2,151.65,75.825000,False
5,9305-CDSKC,Month-to-month,8,820.50,102.562500,False
6,1452-KIOVK,Month-to-month,22,1949.40,88.609091,False
